In [1]:
import os

import folium
import geopandas as gpd
import nivapy3 as nivapy
import pandas as pd
from folium.plugins import MarkerCluster
from tqdm.notebook import tqdm

# Extract data from Vannmiljø

## 1. User options

In [2]:
# Vassdragsområder of interest
vassom_list = range(1, 18)

# Pars of interest. List or None. See
#     https://vannmiljokoder.miljodirektoratet.no/parameter
par_list = [
    "PH",
    "KLFA",
    "N-NH4",
    "N-NO2",
    "N-NO3",
    "N-SNOX",
    "N-TOT",
    "O2",
    "P-ORTO",
    "P-PO4",
    "P-TOT",
    "SI-SIO2",
    "SIO2",
]

# Medium of interest. String or None. See
#     https://vannmiljokoder.miljodirektoratet.no/medium?q=
medium = "VS"

# Output folder
output_dir = r"/home/jovyan/shared/common/JES/vannmiljo_data"

## 2. Get data

In [3]:
names_dict = {
    "WaterLocationID": "station_id",
    "WaterLocationCode": "station_code",
    "Name": "station_name",
    "WaterCategory": "category",
    "CoordX": "utm33_east",
    "CoordY": "utm33_north",
    "FylkeID": "fylke_id",
    "Fylke": "fylke_name",
    "KommuneID": "kommune_id",
    "Kommune": "kommune_name",
    "VassdragsomradeID": "vassom_id",
    "Vassdragsomrade": "vassom_name",
    "VannomradeID": "vannom_id",
    "Vannomrade": "vannom_name",
    "VannregionID": "vannreg_id",
    "Vannregion": "vannreg_name",
    "WaterBodyID": "waterbody_id",
    "WaterBody": "waterbody_name",
    "FeatureType": "feature_type",
    "ActivityID": "activity_id",
    "ActivityName": "activity_name",
    "Employer": "employer",
    "Contractor": "contractor",
    "MediumName": "medium_name",
    "SamplingTime": "sample_date",
    "UpperDepth": "upper_depth",
    "LowerDepth": "lower_depth",
    "FilteredSample": "filtered",
    "ParameterID": "par_id",
    "ParameterName": "par_name",
    "ValueOperator": "flag",
    "RegValue": "value",
    "Unit": "unit",
    "DetectionLimit": "lod",
    "QuantificationLimit": "loq",
}
stn_cols = [
    "station_id",
    "station_code",
    "station_name",
    "feature_type",
    "category",
    "fylke_id",
    "fylke_name",
    "kommune_id",
    "kommune_name",
    "vassom_id",
    "vassom_name",
    "vannom_id",
    "vannom_name",
    "vannreg_id",
    "vannreg_name",
    "waterbody_id",
    "waterbody_name",
    "utm33_east",
    "utm33_north",
]

# Query vassoms individually to avoid overloading the API
df_list = []
for vassom in tqdm(vassom_list, desc="Processing vassom"):
    vassom = f"{vassom:03d}"
    data = {
        "FromRegDate": "1900-01-01",
        "VassdragsomradeIDFilter": [vassom],
        "ParameterIDFilter": par_list,
        "MediumID": medium,
    }
    wc_df = nivapy.da.post_data_to_vannmiljo("GetRegistrations", data=data)
    wc_df = wc_df[names_dict.keys()].rename(columns=names_dict)
    wc_df["sample_date"] = pd.to_datetime(wc_df["sample_date"])
    if len(wc_df) > 0:
        out_csv = os.path.join(output_dir, f"vassom_{vassom}.csv")
        wc_df.to_csv(out_csv, index=False)
        df_list.append(wc_df)

# Combine and save all
wc_df = pd.concat(df_list, axis="rows")
vassom_min = f"{vassom_list[0]:03d}"
vassom_max = f"{vassom_list[-1]:03d}"
out_csv = os.path.join(output_dir, f"vassom_{vassom_min}-{vassom_max}.csv")
wc_df.to_csv(out_csv, index=False)

# Get unique station details
stn_df = wc_df[stn_cols].drop_duplicates(subset="station_code")
stn_gdf = gpd.GeoDataFrame(
    stn_df,
    geometry=gpd.points_from_xy(
        stn_df["utm33_east"], stn_df["utm33_north"], crs="epsg:25833"
    ),
)

Processing vassom:   0%|          | 0/17 [00:00<?, ?it/s]

/tmp/ipykernel_2256/240059336.py:79: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wc_df = pd.concat(df_list, axis="rows")


## 3. Map of stations

In [4]:
# Reproject to WGS84
stn_wgs = stn_gdf.to_crs(epsg=4326)

# Folium map
m = folium.Map(
    location=[stn_wgs.geometry.y.mean(), stn_wgs.geometry.x.mean()], zoom_start=4
)
cluster = MarkerCluster().add_to(m)
for _, row in stn_wgs.iterrows():
    popup_html = f"""
    <b>Station code:</b> {row['station_code']}<br>
    <b>Station name:</b> {row['station_name']}
    """
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(cluster)

m